# Week 6 · Monday: ML Introduction and Supervised Learning

Last week we used large language models that somebody else had trained. Today we train our own, small ones, from examples that come with the right answer. Today:

1. **What a model learns:** features, labels, `fit` and `predict`
2. **Fitting the line:** linear regression, residuals and least squares, MSE and R²
3. **Train and test:** scoring a model on rows it never saw
4. **Types of regression:** many features at once, and curves instead of lines
5. **Classification:** predicting a category, with logistic regression
6. **A real dataset:** breast cancer measurements

Run the next cell first. It holds the plotting helpers we reuse all day, so we can spend class time on the models instead of on matplotlib.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def plot_points(X, labels=None, title="", xlabel="Feature 1", ylabel="Feature 2",
                names=None, centers=None, ax=None):
    """Scatter 2D points, coloured by label if given. Label -1 is drawn as noise (DBSCAN)."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], s=45, edgecolor="black", alpha=0.85)
    else:
        for value in np.unique(labels):
            mask = labels == value
            if value == -1:
                ax.scatter(X[mask, 0], X[mask, 1], s=70, marker="x", color="black", label="Noise (-1)")
            else:
                label = names[value] if names is not None else f"{value}"
                ax.scatter(X[mask, 0], X[mask, 1], s=45, edgecolor="black", alpha=0.85, label=label)
        ax.legend()
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], s=300, marker="X", color="red",
                   edgecolor="black", linewidth=2, label="Centroids")
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if show:
        plt.show()


def plot_decision_boundary(model, X, y, title="", xlabel="Feature 1", ylabel="Feature 2",
                           names=None, ax=None):
    """Colour the background by what the model predicts at every point of a fine grid."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
    grid_predictions = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    classes = np.unique(y)
    colours = ListedColormap(plt.cm.tab10.colors[:len(classes)])   # same colours as the points
    ax.contourf(xx, yy, grid_predictions, levels=np.arange(len(classes) + 1) - 0.5, alpha=0.25, cmap=colours)
    plot_points(X, y, title, xlabel, ylabel, names=names, ax=ax)
    if show:
        plt.show()


def plot_line_chart(xs, ys, title, xlabel, ylabel):
    """A simple line chart with markers, for the elbow and silhouette plots."""
    plt.figure(figsize=(8, 5))
    plt.plot(xs, ys, marker="o", linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(xs)
    plt.grid(alpha=0.3)
    plt.show()


def plot_regression(x, y, model=None, title="", xlabel="Feature 1", ylabel="Target",
                    residuals=False, x_test=None, y_test=None, ax=None):
    """Scatter one feature against the target, draw the model's line or curve, optionally the residuals."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    x, x_test = np.ravel(x), None if x_test is None else np.ravel(x_test)
    ax.scatter(x, y, s=45, edgecolor="black", alpha=0.85, label="Training points" if x_test is not None else "Samples")
    if x_test is not None:
        ax.scatter(x_test, y_test, s=60, marker="s", color="orange", edgecolor="black", label="Test points")
    if model is not None:
        all_x = x if x_test is None else np.concatenate([x, x_test])
        xs = np.linspace(all_x.min(), all_x.max(), 300)
        ax.plot(xs, model.predict(xs.reshape(-1, 1)), color="red", linewidth=3, label="Model")
        if residuals:
            ax.vlines(x, model.predict(x.reshape(-1, 1)), y, color="gray", linewidth=1, label="Residuals")
        all_y = y if y_test is None else np.concatenate([y, y_test])
        margin = 0.15 * (all_y.max() - all_y.min())
        ax.set_ylim(all_y.min() - margin, all_y.max() + margin)   # keep a wild curve from squashing the points
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
    if show:
        plt.show()


print("Setup complete")

## 1. What a model learns

A program you write by hand follows rules you typed. A **machine-learning model** finds its rule from examples whose answers are already known. The vocabulary, as it shows up in code:

| Word | Meaning | In code |
|---|---|---|
| **sample** | one example, one row of the data | one row of `X` |
| **feature** | one input measured for every sample, one column | one column of `X` |
| **label** | the answer we want predicted for each sample (also called the **target**) | `y` |
| **model** | the rule learned from the samples | an object such as `LinearRegression()` |
| **training** | fitting the model to samples whose labels are known | `model.fit(X, y)` |
| **prediction** | the model's answer for a sample | `model.predict(X_new)` |

**scikit-learn** (imported as `sklearn`) is the Python library we use for every model today and tomorrow. Its `make_regression` function makes practice data: one feature, and a label that rises with it plus some random noise.

### 1.1 X is a table, y is a column

In [ ]:
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=100, n_features=1, noise=15, random_state=RANDOM_STATE)

print(f"X shape: {X.shape}   <- (rows, columns): 100 samples, 1 feature")
print(f"y shape: {y.shape}     <- one label per sample")
print("First three samples:")
for features, label in zip(X[:3], y[:3]):
    print(f"  feature {features[0]:6.2f}  ->  label {label:6.1f}")

plot_regression(X, y, title="100 samples: one feature, one label")

- `X` has a capital letter because it is a **table** (2-D: rows and columns), even with only one column. scikit-learn always expects a table for X.
- `y` is a single column of numbers, one per row of X.
- The dots climb from left to right, with scatter around the trend. That scatter is the `noise=15` we asked for. Real data always has some.

### 1.2 Construct, fit, predict

Every scikit-learn model follows the same three steps. Swap in a different algorithm and only the first line changes.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()                    # 1. construct: an empty model
model.fit(X, y)                               # 2. fit: learn the rule from labelled samples

new_samples = np.array([[0.5], [1.0]])        # two new rows, still a table: 2 rows, 1 column
print(f"Predictions: {model.predict(new_samples).round(1)}")   # 3. predict

- `fit(X, y)` is where the learning happens. After it, the model holds numbers it worked out from the data.
- `predict` takes new rows in the same shape as X and returns one answer per row.
- Try changing it: pass `X[:, 0]` (the column on its own, 1-D) to `fit` instead of `X`. scikit-learn stops with `Expected 2D array, got 1D array instead`. `X[:, 0].reshape(-1, 1)` turns the column back into a one-column table. Put `X` back afterwards.

## 2. Fitting the line

`LinearRegression` draws the straight line `y = slope × x + intercept` through the points. **Regression** means predicting a number, as opposed to a category.

### 2.1 Slope and intercept

- The **slope** is how much the prediction goes up for each extra unit of the feature.
- The **intercept** is the prediction when the feature is 0, where the line crosses the y-axis.

For `y = 2x + 5`, the slope is 2 and the intercept is 5. After `fit`, scikit-learn stores them in `coef_` and `intercept_`. A trailing underscore marks a value the model learned from data.

In [ ]:
print(f"Slope (coef_):          {model.coef_[0]:.2f}")
print(f"Intercept (intercept_): {model.intercept_:.2f}")
print(f"Prediction at x = 1:    {model.coef_[0] * 1 + model.intercept_:.2f}   (slope x 1 + intercept)")

- The slope is **45.79**: each extra unit of the feature adds about 46 to the prediction.
- The intercept is **1.75**: the prediction at x = 0.
- Slope × 1 + intercept gives 47.53, which is the 47.5 that `predict` gave for x = 1.0 above. The fitted model is just those two numbers.

### 2.2 Residuals and least squares

Why this line and not a slightly different one? For each sample, the **residual** is the true value minus the prediction: how far the dot is from the line. The plot draws each residual as a grey segment.

In [ ]:
predictions = model.predict(X)
residuals = y - predictions                   # true value minus prediction, one per sample

print(f"First three residuals: {residuals[:3].round(1)}")
print(f"Sum of squared residuals, fitted line:   {np.sum(residuals ** 2):.0f}")
print(f"Sum of squared residuals, line moved up 5: {np.sum((y - (predictions + 5)) ** 2):.0f}")

plot_regression(X, y, model, title="The fitted line and its residuals", residuals=True)

**Least squares** (the method `LinearRegression` uses) picks the line with the **smallest sum of squared residuals**: the smallest total of squared misses.

- Squaring stops misses above the line and below it from cancelling out, and it punishes one big miss more than several small ones.
- Moving the line up by 5 raised the total from 17562 to 20062. Any change to the slope or the intercept raises it. That is what "least" squares means.
- There is a formula for this line, so `fit` computes it in one step. Nothing is tried and corrected here.

### 2.3 Scoring the fit: MSE and R²

Two numbers summarise how well the line fits:

- **Mean squared error (MSE):** the average squared residual. 0 is perfect; the units are the label's units, squared.
- **R²** (R squared, the coefficient of determination): how much of the spread in y the model accounts for. 1.0 is a perfect fit; a model that always predicts the average of y scores 0.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

print(f"MSE:           {mean_squared_error(y, predictions):.1f}")
print(f"R²:            {r2_score(y, predictions):.3f}")
print(f"model.score(): {model.score(X, y):.3f}   <- a regression model's score is R²")

- MSE is **175.6**, the sum of squared residuals (17562) divided by the 100 samples.
- R² is **0.907**: the line accounts for about 91% of the spread in the labels. The rest is the noise.
- `model.score(X, y)` gives the same R², a shortcut every regression model has.

Try changing it: set `noise=60` in the `make_regression` line of section 1.1, then rerun from there down to here. The dots spread far from the line, the grey segments grow, MSE jumps and R² drops well below 0.9. The slope is still positive. Put `noise=15` back and rerun before moving on.

## 3. Train and test

The R² of 0.907 was measured on the same 100 samples the line was fitted to. That is like grading a student on the exact exam questions they practised on: it shows how well they memorised, not how well they will do on new questions.

The fix is to hold some samples back. The **training set** is what the model is fitted on. The **test set** is kept aside and used only to score it, so the score shows how the model does on rows it never saw.

### 3.1 Split, fit on the training rows, score on both

`train_test_split` shuffles the rows and splits them. `test_size=0.2` keeps 20% for testing, and `random_state` makes the shuffle the same on every run. It returns four arrays, always in this order.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Training rows: {X_train.shape}   Test rows: {X_test.shape}")

split_model = LinearRegression().fit(X_train, y_train)     # fitted on the 80 training rows only

print(f"Training R²: {split_model.score(X_train, y_train):.3f}")
print(f"Test R²:     {split_model.score(X_test, y_test):.3f}")

- 80 rows to train on, 20 to test with.
- Training R² is **0.912**, test R² is **0.874**. The test score is usually a little lower, because the line was tuned to the training rows. A small gap like this one is healthy.
- From now on, the **test** score is the number we trust.

Try changing it: set `random_state` in the `train_test_split` line to `0`, then `1`, then `2`. The test R² moves with nothing else changed, because a different 20 rows land in the test set. A fixed seed makes a result repeatable; it does not make it more accurate. Put `RANDOM_STATE` back.

## 4. Types of regression

So far: one feature, one straight line. Real problems have many features, and real relationships bend.

### 4.1 Multiple regression

**Multiple regression** is the same `LinearRegression` with several feature columns. `load_diabetes` is a real medical dataset that ships with scikit-learn: 442 patients, 10 measurements each (age, sex, body-mass index, blood pressure and six blood test results), and as the label, a measure of how far the disease had progressed a year later. The measurements are already rescaled around 0 by the dataset's authors, so don't read them as years or kilograms.

In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X_diabetes, y_diabetes = diabetes.data, diabetes.target
print(f"X shape: {X_diabetes.shape}   y shape: {y_diabetes.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=RANDOM_STATE
)
diabetes_model = LinearRegression().fit(X_train, y_train)

print(f"{len(diabetes_model.coef_)} slopes, one per feature:")
for name, slope in zip(diabetes.feature_names, diabetes_model.coef_):
    print(f"  {name:<4} {slope:8.1f}")
print(f"Intercept:   {diabetes_model.intercept_:.1f}")
print(f"Training R²: {diabetes_model.score(X_train, y_train):.3f}")
print(f"Test R²:     {diabetes_model.score(X_test, y_test):.3f}")

- Ten features give **ten slopes** in `coef_`, in column order, and still **one** intercept.
- Each slope is how much the prediction changes for one unit of that feature, **with the other nine held fixed**. Some features are related to each other (the six blood results especially), and then their slopes can swing a lot from one training set to another, so read them with care.
- Test R² is **0.453**. Ten measurements explain under half of how the disease progressed. That is normal for real data, and far below the 0.874 of our made-up line.

Try changing it: `random_state=0` in the split. Several slopes change, some by a lot, from nothing more than a different set of training rows. Put `RANDOM_STATE` back.

### 4.2 Polynomial regression: degrees 1, 3 and 15

When the points follow a curve, a straight line can't follow them. **Polynomial regression** adds powers of x as extra columns (x², x³, and so on) and fits a plain `LinearRegression` on them. The **degree** is the highest power. Here is what `PolynomialFeatures` does to the single value x = 2 at degree 3:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

print(PolynomialFeatures(degree=3).fit_transform([[2.0]]))     # columns: 1, x, x², x³

One column became four: 1, 2, 4 and 8. The model then learns one coefficient per column, which gives a curve. It is still a linear model, because the prediction is still each column times its coefficient, added up.

Our data: 40 points on a cubic curve (`0.5x³ - x`) with noise added. We hold back a quarter of them as a test set and fit three degrees. `make_pipeline` chains the two steps into one model, so `fit` and `score` run both.

In [ ]:
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(RANDOM_STATE)
x_curve = np.sort(rng.uniform(-3, 3, 40))
y_curve = 0.5 * x_curve**3 - x_curve + rng.normal(0, 2, 40)

X_train, X_test, y_train, y_test = train_test_split(
    x_curve.reshape(-1, 1), y_curve, test_size=0.25, random_state=RANDOM_STATE
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, degree in zip(axes, [1, 3, 15]):
    poly_model = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(X_train, y_train)
    train_r2 = poly_model.score(X_train, y_train)
    test_r2 = poly_model.score(X_test, y_test)
    print(f"Degree {degree:2}: training R² {train_r2:.3f} | test R² {test_r2:.3f}")
    plot_regression(X_train, y_train, poly_model, ax=ax, x_test=X_test, y_test=y_test,
                    title=f"Degree {degree}\ntrain {train_r2:.2f} | test {test_r2:.2f}", xlabel="x", ylabel="y")
plt.tight_layout()
plt.show()

- **Degree 1** **underfits**: a straight line is too simple for a curve, so both scores are poor (training 0.474, test 0.507).
- **Degree 3** matches the true shape and has the best test score, **0.870**.
- **Degree 15** **overfits**: it bends toward individual training points and chases the noise. Its training score is the highest of the three (0.842), and its test score drops to **0.679**. On the plot, the curve wiggles between points and swings away near the ends.

The degree is a **hyperparameter**: a setting a person chooses before fitting, as opposed to the coefficients the model learns. Choose it by the **test** score, never the training score, which rewards a curve for bending toward the very points it is scored on.

Two more types of regression to know by name: **ridge** and **lasso** are regularised linear regression, built to rein in overfitting. They come in week 8.

Try changing it: replace `15` with `20` in the degree list. The curve swings harder near the edges and the test R² falls further, to about 0.42. Push the degree higher and test R² can even go negative, which means the model does worse than always guessing the average. Put `15` back.

## 5. Classification

**Classification** predicts a category instead of a number: spam or not spam, fraud or genuine. Each category is a **class**.

Our first classifier is **logistic regression**. Despite the name, it is a classifier, not a regression model. It is a favourite interview check.

`make_classification` builds practice data: 200 samples, two features (so we can draw them), two classes labelled 0 and 1.

### 5.1 The decision boundary

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_class, y_class = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X_class, y_class, test_size=0.2, random_state=RANDOM_STATE, stratify=y_class
)

classifier = LogisticRegression(max_iter=5000)
classifier.fit(X_train, y_train)

print(f"Training accuracy: {accuracy_score(y_train, classifier.predict(X_train)):.3f}")
print(f"Test accuracy:     {accuracy_score(y_test, classifier.predict(X_test)):.3f}")

plot_decision_boundary(classifier, X_class, y_class, "Logistic regression boundary")

- **Accuracy** (`accuracy_score`) is the fraction of predictions that match the true labels, from 0 to 1. The test accuracy is **0.850**: 34 of the 40 test samples right.
- `stratify=y_class` keeps the same mix of the two classes in the training and test sets.
- The background colour is what the model predicts at each spot. The border between the colours is the **decision boundary**, and for logistic regression it is always a **straight line**. Points on the wrong side of it are the mistakes.
- `max_iter=5000`: `max_iter` caps how many steps the **solver** (the routine that works out the model's numbers) may take. The default of 100 can stop it too early and print a `ConvergenceWarning`. Raising the cap lets the solver finish.

Try changing it: `random_state=7` in the `make_classification` line. A different cloud of points, a boundary at a different angle, and still a straight line. No setting of logistic regression bends it. Put `RANDOM_STATE` back.

### 5.2 A class or a probability

`predict` gives one class per row. `predict_proba` gives the model's **probability** for each class instead: how sure it is.

In [ ]:
first_three = X_test[:3]
print(f"predict:       {classifier.predict(first_three)}")
print("predict_proba: [P(class 0), P(class 1)] per row")
print(classifier.predict_proba(first_three).round(3))
print(f"True labels:   {y_test[:3]}")

- `predict_proba` has one row per sample and **one column per class**. Each row adds up to 1.
- `predict` simply picks the column with the larger probability.
- The first row is 0.951 for class 0: a confident, correct answer.
- The second row is 0.778 for class 0, so `predict` says 0, but its true label is 1. It is one of the six test mistakes, and the model was less sure about it than about the first row. A probability tells you how much to trust each answer; a bare class can't.

## 6. A real dataset: breast cancer

`load_breast_cancer` is a real medical dataset that ships with scikit-learn: 569 tissue samples, 30 numeric measurements each (radius, texture, smoothness and so on), each labelled **malignant** (0) or **benign** (1). Same three steps: construct, fit, predict.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target
print(f"X shape: {X_cancer.shape}")
for name, count in zip(cancer.target_names, np.bincount(y_cancer)):
    print(f"  {name:<9} {count} samples")

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cancer
)
cancer_model = LogisticRegression(max_iter=5000)    # lets the solver finish on unscaled features
cancer_model.fit(X_train, y_train)

print(f"Training accuracy: {accuracy_score(y_train, cancer_model.predict(X_train)):.3f}")
print(f"Test accuracy:     {accuracy_score(y_test, cancer_model.predict(X_test)):.3f}")

- The 30 features sit on very different scales (some below 1, some in the thousands), which is what makes the solver need more steps. `max_iter=5000` lets it finish; tomorrow shows the other fix, scaling the features.
- Training accuracy **0.956**, test accuracy **0.965**. The test score came out slightly higher this time: with only 114 test rows, the luck of the split can do that.

Try changing it: delete `max_iter=5000` so the default of 100 applies. A `ConvergenceWarning` prints above an accuracy that still looks reasonable. A warning is not a crash: it says the solver stopped early. Put `max_iter=5000` back and rerun.

### 6.1 Is 0.965 good?

Before trusting any accuracy, compare it with a "model" that ignores the features and always guesses the most common class.

In [ ]:
always_benign = np.ones_like(y_test)                 # predict class 1 (benign) for every test row
print(f"Always guess benign: {accuracy_score(y_test, always_benign):.3f}")
print(f"Logistic regression: {accuracy_score(y_test, cancer_model.predict(X_test)):.3f}")

- Guessing "benign" every time already scores **0.632**, because about 63% of the samples are benign. So 0.965 is a real improvement, but the starting line was not 0.
- Accuracy can mislead when one class is **rare**. If 10 payments in 1,000 are fraud, a "model" that never says fraud scores 0.990 and catches none of them.
- Accuracy also counts every mistake the same, though calling a malignant sample benign costs far more than the opposite. The metrics that count each kind of mistake separately come in week 8.

## Recap

| Idea or algorithm | Type | Good at | Watch out for |
|---|---|---|---|
| Linear regression | supervised, regression | a number from features; slope and intercept you can read | straight lines only; keeps going past the data's range |
| Polynomial regression | supervised, regression | curved relationships, with the same `LinearRegression` | high degrees overfit: pick the degree by the test score |
| Logistic regression | supervised, classification | a class and a probability for it | a straight boundary only; needs `max_iter` raised (or scaling) on raw features |
| Train/test split | evaluation | an honest score on rows the model never saw | the test score moves with `random_state`; accuracy needs a baseline |

**Quick check**
1. In `model.fit(X, y)`, what is X, what is y, and why must X be 2-D even with one feature?
2. What does least squares minimise?
3. Why do we score a model on a test set instead of on the rows it was trained on?
4. Degree 1 and degree 15 both scored worse on test data than degree 3. What went wrong with each?
5. Logistic regression has "regression" in its name. What does it predict?
6. A fraud model scores 99% accuracy. What would you compare that number with before calling it good?